# LeetCode 3280 — Convert Date to Binary bằng Python

Link đề: https://leetcode.com/problems/convert-date-to-binary/ · Độ khó: **Easy** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def convertDateToBinary(self, date: str) -> str:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn **mọi ngày** từ `1900-01-01` đến `2100-12-31`). Hai ví dụ chính lấy từ đề; ví dụ nào tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho ngày dạng `yyyy-mm-dd`; đổi **năm, tháng, ngày** mỗi phần sang **hệ nhị phân** (không có số `0` đứng đầu), rồi ghép lại theo dạng `năm-tháng-ngày` bằng dấu `-`.
2. Nắm các công cụ Python: **`str.split("-")`** (tách chuỗi), **`int("02")`** (đổi chuỗi thành số, tự bỏ số `0` đầu), **`bin(x)`** và định dạng **`f"{x:b}"`**, **`"-".join(...)`**, `divmod`/`%`/`//` và **`map`**.
3. Đi từ cách **tự đổi nhị phân bằng chia `2` liên tiếp** → **`bin(x)[2:]`** → **định dạng `:b` trong f-string** → **ghép bằng cắt chuỗi theo vị trí cố định** → **dịch bit** (`>>`, `bit_length`).
4. Hiểu mấu chốt: bài **không cần tính toán lịch** gì cả. Ngày đã hợp lệ và có định dạng cố định, ta chỉ cần **tách 3 phần → đổi thành số nguyên (bỏ số `0` đầu) → đổi sang nhị phân → ghép lại**.
5. Tránh các lỗi kinh điển: quên `[2:]` (dính tiền tố `0b`), đổi `bin` trên **chuỗi** chưa `int`, vẫn giữ số `0` đầu (`"02"` phải thành `"10"`, không phải `"010"`), đổi nhị phân **từng chữ số** thay vì cả số, và quên **đảo ngược** các dư khi tự chia `2`.

## 2. Đề bài & ví dụ

Cho chuỗi `date` biểu diễn một ngày theo lịch Gregorian ở dạng `yyyy-mm-dd`.

Ngày này có thể viết ở **dạng nhị phân** bằng cách đổi **năm, tháng, ngày** sang biểu diễn nhị phân **không có số `0` đứng đầu**, rồi viết theo dạng `năm-tháng-ngày`.

Trả về biểu diễn **nhị phân** của `date`.

**Ví dụ 1**

- Input: `date = "2080-02-29"`
- Output: `"100000100000-10-11101"`
- Giải thích: `100000100000`, `10` và `11101` lần lượt là biểu diễn nhị phân của `2080`, `02` (tức `2`) và `29`.

**Ví dụ 2**

- Input: `date = "1900-01-01"`
- Output: `"11101101100-1-1"`
- Giải thích: `11101101100`, `1` và `1` lần lượt là biểu diễn nhị phân của `1900`, `1` và `1`.

**Ràng buộc:** `date.length == 10`; `date[4] == date[7] == '-'` còn mọi ký tự khác là chữ số; đầu vào bảo đảm `date` là một ngày hợp lệ theo lịch Gregorian trong khoảng từ `1900-01-01` đến `2100-12-31` (gồm cả hai mốc).

In [1]:
import random
from datetime import date as Date, timedelta

# (date, kết quả kỳ vọng) — hai ví dụ đầu lấy từ đề
examples = [
    ("2080-02-29", "100000100000-10-11101"),
    ("1900-01-01", "11101101100-1-1"),
]
for d, expected in examples:
    print(f"date={d!r} -> kỳ vọng {expected!r}")

date='2080-02-29' -> kỳ vọng '100000100000-10-11101'
date='1900-01-01' -> kỳ vọng '11101101100-1-1'


## 3. Kiến thức nền

### 3.1. Hệ nhị phân là gì? Đọc và viết số nhị phân

Hệ **thập phân** dùng 10 chữ số `0..9`, mỗi vị trí có giá trị gấp `10` lần vị trí bên phải (`1`, `10`, `100`, ...). Hệ **nhị phân** chỉ dùng `0` và `1`; mỗi vị trí có giá trị gấp `2` lần vị trí bên phải: `1, 2, 4, 8, 16, 32, ...`.

Ví dụ `29 = 16 + 8 + 4 + 1`, nên bật các vị trí `16, 8, 4, 1` và tắt vị trí `2`: viết `11101`. Đọc ngược: `1*16 + 1*8 + 1*4 + 0*2 + 1*1 = 29`.

In [2]:
bits = "11101"
total = 0
for i, ch in enumerate(reversed(bits)):          # vị trí 0 là bit phải nhất
    weight = 2 ** i
    print(f"bit {ch} ở vị trí {i} có giá trị {weight}", "-> cộng" if ch == "1" else "-> bỏ")
    if ch == "1":
        total += weight
print(total)                                     # 29
print(int(bits, 2))                              # 29: Python tự đọc nhị phân

bit 1 ở vị trí 0 có giá trị 1 -> cộng
bit 0 ở vị trí 1 có giá trị 2 -> bỏ
bit 1 ở vị trí 2 có giá trị 4 -> cộng
bit 1 ở vị trí 3 có giá trị 8 -> cộng
bit 1 ở vị trí 4 có giá trị 16 -> cộng
29
29


### 3.2. Tách chuỗi bằng `split`

`"2080-02-29".split("-")` cắt chuỗi tại mỗi dấu `-` và trả về **danh sách các chuỗi**: `["2080", "02", "29"]`. Có thể gán ngay vào ba biến: `year, month, day = date.split("-")`.

In [3]:
date = "2080-02-29"
parts = date.split("-")
print(parts, len(parts))                         # ['2080', '02', '29'] 3
year, month, day = date.split("-")
print(year, month, day)
print(type(year))                                # <class 'str'> -- vẫn là chuỗi

['2080', '02', '29'] 3
2080 02 29
<class 'str'>


### 3.3. `int("02")` bỏ số `0` đứng đầu

Đổi chuỗi sang số nguyên bằng `int(...)`. Khi đổi, số `0` đứng đầu **biến mất** vì với số nguyên `02` và `2` là một. Đây chính là cách ta thỏa yêu cầu "không có số `0` đứng đầu" của đề.

In [4]:
print(int("02"), int("2080"), int("0007"), int("10"))     # 2 2080 7 10
print(int("02") == 2, "02" == "2")                          # True False

2 2080 7 10
True False


### 3.4. `bin` và định dạng `:b`

`bin(29)` trả về chuỗi `'0b11101'` — có **tiền tố `0b`** báo "đây là số nhị phân". Muốn bỏ nó, cắt từ ký tự thứ `2` trở đi: `bin(29)[2:]`. Cách khác là **định dạng**: `f"{29:b}"` hoặc `format(29, "b")` cho thẳng `'11101'` không có tiền tố. `bin` chỉ nhận **số**; truyền chuỗi `"29"` sẽ lỗi.

In [5]:
x = 29
print(bin(x))                    # 0b11101
print(bin(x)[2:])                # 11101
print(f"{x:b}", format(x, "b"))  # 11101 11101
print(f"{x:08b}")                # 00011101  (thêm số 0 đầu cho đủ 8 chữ số -- KHÔNG dùng cho bài này)
try:
    bin("29")
except TypeError as e:
    print("TypeError:", e)

0b11101
11101
11101 11101
00011101
TypeError: 'str' object cannot be interpreted as an integer


### 3.5. `join` và `map`

`"-".join(["a", "b", "c"])` ghép các chuỗi trong danh sách, chèn `-` ở giữa: `"a-b-c"`. `join` chỉ nhận các **chuỗi**. `map(f, danh_sách)` áp hàm `f` lên từng phần tử (trả về một "dòng" phần tử, thường bọc trong `list(...)`).

In [6]:
print("-".join(["100", "10", "11"]))                   # 100-10-11
print(list(map(int, ["2080", "02", "29"])))            # [2080, 2, 29]
print(list(map(lambda v: f"{v:b}", [2080, 2, 29])))    # ['100000100000', '10', '11101']

100-10-11
[2080, 2, 29]
['100000100000', '10', '11101']


### 3.6. `//`, `%` và `divmod`

`a // 2` là **chia nguyên** (bỏ phần thập phân), `a % 2` là **dư**. `divmod(a, 2)` trả cả hai: `(a // 2, a % 2)`. Chia `2` liên tiếp và ghi lại các dư chính là cách đổi nhị phân bằng tay (dư của lần chia đầu là bit **phải nhất**).

In [7]:
print(29 // 2, 29 % 2, divmod(29, 2))     # 14 1 (14, 1)
n = 29
while n > 0:
    n, r = divmod(n, 2)
    print("còn lại", n, "dư", r)

14 1 (14, 1)
còn lại 14 dư 1
còn lại 7 dư 0
còn lại 3 dư 1
còn lại 1 dư 1
còn lại 0 dư 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài này không bắt ta xử lý lịch: ngày **luôn hợp lệ** và luôn ở dạng cố định `yyyy-mm-dd`. Công việc gói gọn trong ba bước:

1. **Tách** `"2080-02-29"` thành `"2080"`, `"02"`, `"29"`.
2. **Đổi** mỗi phần thành số nguyên (`2080`, `2`, `29`) — việc này loại số `0` đầu.
3. **Đổi nhị phân** từng số và **ghép** lại bằng `-`.

Tính tay `29` bằng cách chia `2` liên tiếp:

| bước | số hiện tại | `// 2` | dư (`% 2`) |
|---|---|---|---|
| 1 | 29 | 14 | 1 |
| 2 | 14 | 7 | 0 |
| 3 | 7 | 3 | 1 |
| 4 | 3 | 1 | 1 |
| 5 | 1 | 0 | 1 |

Đọc các dư **từ dưới lên**: `1 1 1 0 1` ⇒ `11101`. Làm tương tự `2` ⇒ `10` và `2080` ⇒ `100000100000` (bạn thử kiểm bằng `2048 + 32 = 2080`).

In [8]:
date = "2080-02-29"
year, month, day = (int(p) for p in date.split("-"))
print(year, month, day)                                   # 2080 2 29
print(f"{year:b}-{month:b}-{day:b}")                      # 100000100000-10-11101
print(2048 + 32)                                          # 2080 = 2^11 + 2^5

2080 2 29
100000100000-10-11101
2080


## 5. Cách 1 — Brute Force / baseline: tự đổi nhị phân bằng chia `2` liên tiếp

**Ý tưởng:** không dùng `bin`. Viết hàm `to_binary_manual(n)`: lặp `n, r = divmod(n, 2)`, ghi `r` vào danh sách đến khi `n = 0`; các dư xuất hiện từ bit **phải sang trái** nên cuối cùng **đảo danh sách** rồi ghép thành chuỗi. Với `n = 0` vòng lặp không chạy (ngày/tháng không bao giờ là `0` nên bài này không cần, nhưng nên nhớ).

In [9]:
def to_binary_manual(n: int) -> str:
    if n == 0:
        return "0"
    digits = []
    while n > 0:
        n, r = divmod(n, 2)
        digits.append(str(r))         # dư đầu tiên là bit phải nhất
    digits.reverse()                  # đảo để bit trái nhất lên đầu
    return "".join(digits)


def convert_manual(date: str) -> str:
    parts = date.split("-")
    return "-".join(to_binary_manual(int(p)) for p in parts)


print(to_binary_manual(29), to_binary_manual(2), to_binary_manual(2080))   # 11101 10 100000100000
print(convert_manual("2080-02-29"), convert_manual("1900-01-01"))

11101 10 100000100000
100000100000-10-11101 11101101100-1-1


**Độ phức tạp (Cách 1).** Mỗi số `n` cần `O(log n)` lần chia (số bit). Với năm `<= 2100` cần tối đa `12` lần, tháng `<= 12` tối đa `4` lần, ngày `<= 31` tối đa `5` lần — tổng chỉ khoảng `21` bước ⇒ **Time `O(1)`**, **Space `O(1)`** (chuỗi kết quả dài tối đa `12 + 4 + 5 + 2 = 23`). **Nút thắt:** hoàn toàn đúng, nhưng Python đã có sẵn công cụ đổi nhị phân; ta tự viết lại là thừa và dễ sai (quên đảo, quên `n = 0`).

## 6. Ý tưởng thuật toán / bước đột phá

**Python có sẵn `bin` / định dạng `:b`.** Chỉ cần ba bước ở Mục 4: `split` → `int` → `bin`/`:b` → `join`. Điểm tinh tế duy nhất là yêu cầu **không có số `0` đứng đầu**: ta đạt được nhờ `int("02") = 2`, và `bin(2) = '0b10'`, `bin(2)[2:] = '10'` (hoặc `f"{2:b}" = '10'`). Cả hai cách đều không thêm số `0` đầu.

Nói gọn: **`"-".join(f"{int(p):b}" for p in date.split("-"))`** — một dòng, chạy `O(1)`.

In [10]:
date = "2080-02-29"
steps = [int(p) for p in date.split("-")]
print(steps)                                             # [2080, 2, 29]
print([bin(v) for v in steps])                           # có tiền tố 0b
print([bin(v)[2:] for v in steps])                       # đã bỏ tiền tố
print("-".join(bin(v)[2:] for v in steps))               # 100000100000-10-11101

[2080, 2, 29]
['0b100000100000', '0b10', '0b11101']
['100000100000', '10', '11101']
100000100000-10-11101


## 7. Các cách tiếp theo

### Cách 2 — `split` + `int` + `bin(...)[2:]` + `join`

Giải thích từng đoạn:

- `date.split("-")` tách thành ba chuỗi `year`, `month`, `day`.
- `int(p)` đổi mỗi chuỗi thành số (bỏ số `0` đầu).
- `bin(...)[2:]` đổi thành chuỗi nhị phân và bỏ tiền tố `0b`.
- `"-".join(...)` ghép lại với `-`.

In [11]:
def convert_bin(date: str) -> str:
    return "-".join(bin(int(p))[2:] for p in date.split("-"))


class Solution2:
    def convertDateToBinary(self, date: str) -> str:
        year, month, day = date.split("-")
        return "-".join([bin(int(year))[2:], bin(int(month))[2:], bin(int(day))[2:]])


print(convert_bin("2080-02-29"), convert_bin("1900-01-01"))
print(Solution2().convertDateToBinary("2080-02-29"), Solution2().convertDateToBinary("1900-01-01"))

100000100000-10-11101 11101101100-1-1
100000100000-10-11101 11101101100-1-1


**Độ phức tạp Cách 2:** **Time `O(1)`**, **Space `O(1)`** (độ dài đầu vào cố định `10`).

### Cách 3 — Định dạng `f"{x:b}"` (gọn nhất)

`f"{x:b}"` đổi `x` sang nhị phân và **không có tiền tố** nên khỏi cần `[2:]`. Đây là cách nên dùng hằng ngày.

In [12]:
def convert_fstring(date: str) -> str:
    year, month, day = map(int, date.split("-"))
    return f"{year:b}-{month:b}-{day:b}"


class Solution3:
    def convertDateToBinary(self, date: str) -> str:
        return "-".join(f"{int(p):b}" for p in date.split("-"))


print(convert_fstring("2080-02-29"), convert_fstring("1900-01-01"))
print(Solution3().convertDateToBinary("2080-02-29"), Solution3().convertDateToBinary("1900-01-01"))

100000100000-10-11101 11101101100-1-1
100000100000-10-11101 11101101100-1-1


`year, month, day = map(int, date.split("-"))` gán ba số vào ba biến cùng lúc (`map` trả về dòng ba số). Vì ngày hợp lệ và luôn có đúng `3` phần nên phép gán không bao giờ lỗi.

**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Cắt theo vị trí cố định thay cho `split`

Đề cho biết `date.length == 10` và dấu `-` ở vị trí `4` và `7`. Do đó năm là `date[0:4]`, tháng là `date[5:7]`, ngày là `date[8:10]` (hay `date[8:]`). Cách này **không cần `split`**, nhưng nếu định dạng khác (ví dụ ngày không có số `0` đầu) thì hỏng, còn `split` thì vẫn đúng. Ta kết hợp `format(x, "b")` thay cho f-string để làm quen với hàm `format`.

In [13]:
def convert_slice(date: str) -> str:
    year = int(date[0:4])
    month = int(date[5:7])
    day = int(date[8:])
    return format(year, "b") + "-" + format(month, "b") + "-" + format(day, "b")


class Solution4:
    def convertDateToBinary(self, date: str) -> str:
        return "-".join(format(int(date[a:b]), "b") for a, b in ((0, 4), (5, 7), (8, 10)))


print(convert_slice("2080-02-29"), convert_slice("1900-01-01"))
print(Solution4().convertDateToBinary("2080-02-29"), Solution4().convertDateToBinary("1900-01-01"))

100000100000-10-11101 11101101100-1-1
100000100000-10-11101 11101101100-1-1


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 5 — Dịch bit (`>>`) và `bit_length`: hiểu nhị phân "từ bên trong"

Số `n` có `n.bit_length()` bit. Bit thứ `i` (đếm từ phải, bắt đầu `0`) là `(n >> i) & 1`: dịch `n` sang phải `i` vị trí rồi lấy bit cuối. Duyệt `i` từ **bit cao nhất xuống `0`** ta được chuỗi nhị phân **không có số `0` đầu** ngay từ đầu, không cần đảo.

In [14]:
def to_binary_bits(n: int) -> str:
    chars = []
    for i in range(n.bit_length() - 1, -1, -1):     # từ bit cao nhất xuống bit 0
        chars.append("1" if (n >> i) & 1 else "0")
    return "".join(chars)


def convert_bits(date: str) -> str:
    return "-".join(to_binary_bits(int(p)) for p in date.split("-"))


print((29).bit_length(), (2080).bit_length())              # 5 12
print(29 >> 2, (29 >> 2) & 1)                              # 7 1
print(to_binary_bits(29), to_binary_bits(2080))
print(convert_bits("2080-02-29"), convert_bits("1900-01-01"))

5 12
7 1
11101 100000100000
100000100000-10-11101 11101101100-1-1


**Độ phức tạp Cách 5:** **Time `O(1)`** (tối đa `12` bit mỗi số), **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 1 (chia `2` liên tiếp) trên `29` và cả ngày `"2080-02-29"`, rồi Cách 5 (dịch bit) trên `2`.

In [15]:
def trace_manual(n: int) -> str:
    print(f"đổi {n} sang nhị phân bằng chia 2 liên tiếp")
    digits = []
    step = 0
    while n > 0:
        step += 1
        old = n
        n, r = divmod(n, 2)
        digits.append(str(r))
        print(f"  bước {step}: {old} // 2 = {n}, dư = {r}, các dư so far = {digits}")
    digits.reverse()
    result = "".join(digits)
    print(f"  đảo lại -> {result}")
    return result


assert trace_manual(29) == "11101"
assert trace_manual(2) == "10"

đổi 29 sang nhị phân bằng chia 2 liên tiếp
  bước 1: 29 // 2 = 14, dư = 1, các dư so far = ['1']
  bước 2: 14 // 2 = 7, dư = 0, các dư so far = ['1', '0']
  bước 3: 7 // 2 = 3, dư = 1, các dư so far = ['1', '0', '1']
  bước 4: 3 // 2 = 1, dư = 1, các dư so far = ['1', '0', '1', '1']
  bước 5: 1 // 2 = 0, dư = 1, các dư so far = ['1', '0', '1', '1', '1']
  đảo lại -> 11101
đổi 2 sang nhị phân bằng chia 2 liên tiếp
  bước 1: 2 // 2 = 1, dư = 0, các dư so far = ['0']
  bước 2: 1 // 2 = 0, dư = 1, các dư so far = ['0', '1']
  đảo lại -> 10


In [16]:
def trace_date(date: str) -> str:
    print(f"date = {date!r}")
    parts = date.split("-")
    print(f"  split -> {parts}")
    numbers = [int(p) for p in parts]
    print(f"  int   -> {numbers}  (số 0 đầu đã mất)")
    bins = [f"{v:b}" for v in numbers]
    print(f"  :b    -> {bins}")
    result = "-".join(bins)
    print(f"  join  -> {result!r}")
    return result


assert trace_date("2080-02-29") == "100000100000-10-11101"
assert trace_date("1900-01-01") == "11101101100-1-1"

date = '2080-02-29'
  split -> ['2080', '02', '29']
  int   -> [2080, 2, 29]  (số 0 đầu đã mất)
  :b    -> ['100000100000', '10', '11101']
  join  -> '100000100000-10-11101'
date = '1900-01-01'
  split -> ['1900', '01', '01']
  int   -> [1900, 1, 1]  (số 0 đầu đã mất)
  :b    -> ['11101101100', '1', '1']
  join  -> '11101101100-1-1'


In [17]:
def trace_bits(n: int) -> str:
    print(f"n = {n}, bit_length = {n.bit_length()}")
    chars = []
    for i in range(n.bit_length() - 1, -1, -1):
        bit = (n >> i) & 1
        chars.append(str(bit))
        print(f"  i = {i}: n >> {i} = {n >> i}, & 1 = {bit}, chuỗi = {''.join(chars)}")
    return "".join(chars)


assert trace_bits(2) == "10"
assert trace_bits(13) == "1101"

n = 2, bit_length = 2
  i = 1: n >> 1 = 1, & 1 = 1, chuỗi = 1
  i = 0: n >> 0 = 2, & 1 = 0, chuỗi = 10
n = 13, bit_length = 4
  i = 3: n >> 3 = 1, & 1 = 1, chuỗi = 1
  i = 2: n >> 2 = 3, & 1 = 1, chuỗi = 11
  i = 1: n >> 1 = 6, & 1 = 0, chuỗi = 110
  i = 0: n >> 0 = 13, & 1 = 1, chuỗi = 1101


Bảng trace Cách 2/3 cho `"2080-02-29"`:

| phần | chuỗi | `int` | nhị phân |
|---|---|---|---|
| năm | `"2080"` | `2080` | `100000100000` |
| tháng | `"02"` | `2` | `10` |
| ngày | `"29"` | `29` | `11101` |

Ghép bằng `-`: `100000100000-10-11101` — đúng với ví dụ 1 của đề.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 nhưng viết rõ ba biến để dễ đọc.

In [18]:
class Solution:
    def convertDateToBinary(self, date: str) -> str:
        year, month, day = date.split("-")
        parts = [int(year), int(month), int(day)]        # đổi sang số: tự bỏ số 0 đầu
        return "-".join(f"{p:b}" for p in parts)          # :b = nhị phân không tiền tố


sol = Solution()
print(sol.convertDateToBinary("2080-02-29"))   # 100000100000-10-11101
print(sol.convertDateToBinary("1900-01-01"))   # 11101101100-1-1

100000100000-10-11101
11101101100-1-1


## 10. Vì sao thuật toán đúng

- **Tách đúng:** đề đảm bảo đúng hai dấu `-` ở vị trí `4`, `7` nên `split("-")` cho đúng `3` phần: năm, tháng, ngày.
- **Không có số `0` đầu:** `int("02")` ra `2`; biểu diễn nhị phân chuẩn của số nguyên dương **luôn bắt đầu bằng `1`** (`bin` và `:b` không thêm số `0` đầu). Vậy kết quả mỗi phần thỏa yêu cầu.
- **Đúng giá trị:** `:b` đổi số `n` thành chuỗi bit sao cho `Σ bit_i * 2^i = n` (kiểm tra bằng `int(chuỗi, 2) == n`).
- **Ghép đúng thứ tự:** `join` giữ nguyên thứ tự `năm-tháng-ngày` của danh sách `split`.
- **Kiểm chứng thực nghiệm:** Mục 12 đối chiếu mọi ngày từ `1900-01-01` đến `2100-12-31` và kiểm tra bằng **phép đảo ngược** (đọc lại `int(phần, 2)` rồi ghép về `yyyy-mm-dd`).

## 11. Phân tích Time / Space Complexity

Đầu vào luôn dài `10` ký tự và giá trị tối đa `2100`, nên mọi cách đều là hằng số.

- **Time:** `O(1)`. Nếu tính theo giá trị `V` của các số (năm), chia nhị phân tốn `O(log V)` bước; ở đây `log2(2100) ≈ 11`.
- **Space:** `O(1)` — danh sách ba phần tử và chuỗi kết quả dài tối đa `23` ký tự.
- **Cách 1 và Cách 5** tốn thêm vài bước lặp nhưng vẫn `O(1)`. **Cách 2, 3, 4** dựa vào thư viện chuẩn (chạy bằng C bên dưới) nên nhanh hơn trong thực tế.

## 12. Bộ test

Gom các cách vào `impls`, so với kỳ vọng. Với bài toán này ta có **chuẩn kiểm tra độc lập**: đảo ngược kết quả về lại `yyyy-mm-dd` bằng `binary_to_date` và so với đầu vào.

In [19]:
def binary_to_date(binary: str) -> str:
    """Đảo ngược: 'năm_bin-tháng_bin-ngày_bin' -> 'yyyy-mm-dd'."""
    y, m, d = (int(p, 2) for p in binary.split("-"))
    return f"{y:04d}-{m:02d}-{d:02d}"


def is_valid_answer(date: str, binary: str) -> bool:
    """Kiểm tra hợp lệ độc lập: 3 phần, mỗi phần chỉ 0/1, bắt đầu bằng '1', đọc lại đúng ngày cũ."""
    parts = binary.split("-")
    return (
        len(parts) == 3
        and all(p and set(p) <= {"0", "1"} and p[0] == "1" for p in parts)
        and binary_to_date(binary) == date
    )


sol = Solution()
impls = {
    "manual": convert_manual,
    "bin": convert_bin,
    "fstring": convert_fstring,
    "slice": convert_slice,
    "bits": convert_bits,
    "sol2": Solution2().convertDateToBinary,
    "sol3": Solution3().convertDateToBinary,
    "sol4": Solution4().convertDateToBinary,
    "leetcode": sol.convertDateToBinary,
}


def check_all(date, expected, label):
    for key, fn in impls.items():
        actual = fn(date)
        assert actual == expected, (label, key, date, actual, expected)
        assert is_valid_answer(date, actual), (label, key, date, actual)
    print("PASS", label)


for idx, (d, expected) in enumerate(examples, 1):
    check_all(d, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Ngày đầu và cuối khoảng cho phép, tháng/ngày có số `0` đầu, tháng `12`, ngày `31`, năm nhuận (đều tự dựng; kết quả kiểm bằng `bin` và phép đảo ngược).

In [20]:
check_all("1900-01-01", "11101101100-1-1", "ngày đầu khoảng cho phép")
check_all("2100-12-31", "100000110100-1100-11111", "ngày cuối khoảng cho phép")
check_all("2000-02-29", "11111010000-10-11101", "năm nhuận 2000")
check_all("1999-10-10", "11111001111-1010-1010", "tháng và ngày hai chữ số")
check_all("2024-09-05", "11111101000-1001-101", "tháng, ngày có số 0 đầu")
check_all("2048-08-08", "100000000000-1000-1000", "năm là lũy thừa của 2 (2048 = 2^11)")
check_all("1950-04-30", "11110011110-100-11110", "ngày 30")

PASS ngày đầu khoảng cho phép
PASS ngày cuối khoảng cho phép
PASS năm nhuận 2000
PASS tháng và ngày hai chữ số
PASS tháng, ngày có số 0 đầu
PASS năm là lũy thừa của 2 (2048 = 2^11)
PASS ngày 30


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "gần đúng": (a) quên `[2:]` nên dính `0b`, (b) đổi **từng chữ số** thay vì cả số, (c) **giữ số `0` đầu** của tháng/ngày bằng định dạng `:04b`. Ta kiểm tra chúng sai trên ví dụ đề.

In [21]:
def wrong_prefix(date):
    return "-".join(bin(int(p)) for p in date.split("-"))

def wrong_per_digit(date):
    return "".join(f"{int(ch):b}" if ch != "-" else "-" for ch in date)

def wrong_leading_zero(date):
    return "-".join(f"{int(p):012b}" for p in date.split("-"))


for name, wrong in [("quên [2:]", wrong_prefix), ("từng chữ số", wrong_per_digit), ("giữ số 0 đầu", wrong_leading_zero)]:
    got = wrong("2080-02-29")
    print(f"{name}: {got!r} != {'100000100000-10-11101'!r} ->", got != "100000100000-10-11101")
    assert got != "100000100000-10-11101"

quên [2:]: '0b100000100000-0b10-0b11101' != '100000100000-10-11101' -> True
từng chữ số: '10010000-010-101001' != '100000100000-10-11101' -> True
giữ số 0 đầu: '100000100000-000000000010-000000011101' != '100000100000-10-11101' -> True


### 12.3. Vét cạn mọi ngày hợp lệ và đối chiếu ngẫu nhiên (có seed)

Ta duyệt **mọi ngày** từ `1900-01-01` đến `2100-12-31` bằng `datetime.date` (thư viện chuẩn), tạo chuỗi `yyyy-mm-dd` bằng `isoformat()`, và so sánh toàn bộ cách làm với Cách 2 (`bin`). Ngoài ra kiểm tra `is_valid_answer` cho mỗi kết quả và thử thêm `5000` ngày ngẫu nhiên (seed cố định).

In [22]:
start, end = Date(1900, 1, 1), Date(2100, 12, 31)
day = start
count = 0
while day <= end:
    s = day.isoformat()
    ref = convert_bin(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
    assert is_valid_answer(s, ref), s
    count += 1
    day += timedelta(days=1)
print("PASS vét cạn", count, "ngày")

rng = random.Random(3280)
total_days = (end - start).days
for _ in range(5000):
    s = (start + timedelta(days=rng.randint(0, total_days))).isoformat()
    assert sol.convertDateToBinary(s) == convert_manual(s)
print("PASS 5000 ngày ngẫu nhiên")

PASS vét cạn 73414 ngày
PASS 5000 ngày ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên bỏ tiền tố `0b`

In [23]:
def sai_tien_to(date):
    return "-".join(bin(int(p)) for p in date.split("-"))

print(sai_tien_to("2080-02-29"))     # 0b100000100000-0b10-0b11101

0b100000100000-0b10-0b11101


**Vì sao sai:** `bin(x)` luôn trả về chuỗi bắt đầu bằng `0b`. Cần cắt `[2:]` hoặc dùng `f"{x:b}"`.

### Lỗi 2: Gọi `bin` trên chuỗi (chưa `int`)

In [24]:
def sai_quen_int(date):
    return "-".join(bin(p)[2:] for p in date.split("-"))

try:
    sai_quen_int("2080-02-29")
except TypeError as e:
    print("TypeError:", e)

TypeError: 'str' object cannot be interpreted as an integer


**Vì sao sai:** `date.split("-")` cho các **chuỗi** (`"2080"`, ...). `bin` cần số nguyên, nên phải `int(p)` trước. Hơn nữa `int` còn giúp bỏ số `0` đầu.

### Lỗi 3: Đổi nhị phân từng chữ số riêng lẻ

In [25]:
def sai_tung_chu_so(date):
    return "-".join("".join(f"{int(ch):b}" for ch in p) for p in date.split("-"))

print(sai_tung_chu_so("2080-02-29"))                         # sai: ghép 10 + 0 + 1000 + 0 ...
print(sai_tung_chu_so("2080-02-29") == "100000100000-10-11101")

10010000-010-101001
False


**Vì sao sai:** biểu diễn nhị phân là của **cả số** `2080`, không phải ghép nhị phân của `2`, `0`, `8`, `0`. Số `2080` có giá trị khác hẳn dãy các chữ số của nó.

### Lỗi 4: Tự chia `2` nhưng quên đảo thứ tự

In [26]:
def sai_quen_dao(n):
    digits = []
    while n > 0:
        n, r = divmod(n, 2)
        digits.append(str(r))
    return "".join(digits)           # thiếu digits.reverse()

print(sai_quen_dao(29), "(đúng là 11101)")      # 10111 -- ngược
print(to_binary_manual(29))
assert sai_quen_dao(29) != "11101" and to_binary_manual(29) == "11101"

10111 (đúng là 11101)
11101


**Vì sao sai:** dư đầu tiên là bit **thấp nhất** (bên phải), nên chuỗi dư phải được đọc **từ cuối về đầu**. Hãy `reverse()` (hoặc `digits[::-1]`), hoặc chèn mỗi dư vào **đầu** chuỗi.

**Code đúng (nhắc lại):** `"-".join(f"{int(p):b}" for p in date.split("-"))`.

In [27]:
def dung(date):
    return "-".join(f"{int(p):b}" for p in date.split("-"))

for d, expected in examples:
    assert dung(d) == expected
print("PASS code đúng")

PASS code đúng


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đổi sang nhị phân**" trong Python: `bin(x)[2:]`, `f"{x:b}"`, `format(x, "b")`. Ngược lại: `int(s, 2)`. Hệ khác: `:o` (bát phân), `:x` (thập lục phân).
- "**Không có số `0` đứng đầu**": đổi chuỗi sang `int` trước. Muốn **thêm** số `0` đầu cho đủ độ rộng thì dùng `:04d` hoặc `:08b`.
- Đề dạng "**định dạng cố định**" (`yyyy-mm-dd`): `split("-")` hoặc cắt chuỗi, **không** cần thư viện lịch.
- Với bài "đổi từng phần rồi ghép lại" nghĩ ngay tới `join` + generator: `sep.join(f(p) for p in s.split(sep))`.
- Khi nghi ngờ, kiểm tra bằng **phép đảo ngược** (`int(x, 2)`): chuyển đi rồi chuyển về phải ra đúng đầu vào.

## 15. Biến thể của bài

1. **Đảo ngược:** từ `"100000100000-10-11101"` về `"2080-02-29"` (có số `0` đầu cho tháng/ngày).
2. **Hệ khác:** đổi sang bát phân hoặc thập lục phân (`:o`, `:x`).
3. **Đếm số bit `1`** (popcount) ở mỗi phần: `bin(x).count("1")`.
4. **Độ rộng cố định:** năm luôn `12` bit, tháng `4` bit, ngày `5` bit — thêm số `0` đầu bằng `:012b`, `:04b`, `:05b`.
5. **Ngày thứ mấy trong năm** (day-of-year) rồi đổi sang nhị phân (dùng `datetime.date.timetuple().tm_yday`).

In [28]:
def binary_to_date_v(binary):
    y, m, d = (int(p, 2) for p in binary.split("-"))
    return f"{y:04d}-{m:02d}-{d:02d}"


def date_to_base(date, spec):
    """spec: 'b' nhị phân, 'o' bát phân, 'x' thập lục phân."""
    return "-".join(format(int(p), spec) for p in date.split("-"))


def popcounts(date):
    return [bin(int(p)).count("1") for p in date.split("-")]


def fixed_width(date):
    y, m, d = map(int, date.split("-"))
    return f"{y:012b}-{m:04b}-{d:05b}"


def day_of_year_binary(date):
    y, m, d = map(int, date.split("-"))
    return f"{Date(y, m, d).timetuple().tm_yday:b}"


print(binary_to_date_v("100000100000-10-11101"))        # 2080-02-29
print(date_to_base("2080-02-29", "o"), date_to_base("2080-02-29", "x"))     # 4040-2-35 820-2-1d
print(popcounts("2080-02-29"))                          # [2, 1, 4]
print(fixed_width("1900-01-01"))                        # 011101101100-0001-00001
print(day_of_year_binary("2080-02-29"))                 # 111100  (ngày thứ 60 của năm nhuận)

2080-02-29
4040-2-35 820-2-1d
[2, 1, 4]
011101101100-0001-00001
111100


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chia `2` liên tiếp | tự đổi nhị phân, nhớ đảo | `O(1)` | `O(1)` | học cơ chế đổi hệ cơ số |
| 2. `bin(int(p))[2:]` | `split` + `bin` + bỏ `0b` | `O(1)` | `O(1)` | quen thuộc, dễ nhớ |
| 3. `f"{int(p):b}"` | định dạng nhị phân | `O(1)` | `O(1)` | **nên nộp**, gọn và không tiền tố |
| 4. Cắt chuỗi cố định | `date[0:4]`, `date[5:7]`, `date[8:]` | `O(1)` | `O(1)` | khi chắc chắn định dạng cố định |
| 5. Dịch bit | `(n >> i) & 1`, `bit_length` | `O(1)` | `O(1)` | hiểu bit, bài bit manipulation |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"2024-09-05".split("-")`, `int("05")`, `bin(5)`, `f"{5:b}"`, `Solution().convertDateToBinary("2000-01-01")`.

**Bài 2 (tính tay).** Đổi `"1999-12-25"` bằng cách chia `2` liên tiếp cho `12` và `25` (với năm `1999` hãy thử tách thành các lũy thừa của `2`, bắt đầu từ lũy thừa lớn nhất không vượt quá `1999`, rồi điền các vị trí bit).

**Bài 3 (điền TODO).** Hoàn thành `convert_todo(date)` theo Cách 3.

In [29]:
def convert_todo(date):
    # TODO 1: tách date thành list các chuỗi bằng split("-")
    # TODO 2: với mỗi chuỗi p, đổi sang int rồi sang nhị phân (f"{...:b}")
    # TODO 3: ghép các phần bằng "-" và trả về
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `convert_recursive(date)` trong đó phần đổi nhị phân của một số được cài bằng **đệ quy**: `to_bin(n) = to_bin(n // 2) + str(n % 2)` (với `n < 2` trả thẳng `str(n)`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `binary_to_date(binary)` đảo ngược kết quả bài (có số `0` đầu cho tháng và ngày: `yyyy-mm-dd`). (b) Viết `is_palindrome_binary_date(date)` trả `True` nếu **chuỗi nhị phân của năm** là đối xứng (đọc xuôi ngược giống nhau).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `['2024', '09', '05']`, `5`, `'0b101'`, `'101'`, `'11111010000-1-1'`. Bài 3: `"-".join(f"{int(p):b}" for p in date.split("-"))`.
- **Mức 2 (vừa):** Bài 2: tháng `12 = 8 + 4` ⇒ `1100`; ngày `25 = 16 + 8 + 1` ⇒ `11001`; năm `1999`: bật các bit `1024, 512, 256, 128, 64, 8, 4, 2, 1` ⇒ `11111001111`. Bài 4: `if n < 2: return str(n)`.
- **Mức 3 (sâu):** Bài 5(a): `int(p, 2)` đọc nhị phân, rồi `f"{y:04d}-{m:02d}-{d:02d}"` để thêm số `0` đầu. 5(b): lấy phần đầu `year_bin = f"{int(date[:4]):b}"` rồi so với `year_bin[::-1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert "2024-09-05".split("-") == ["2024", "09", "05"]
assert int("05") == 5
assert bin(5) == "0b101"
assert f"{5:b}" == "101"
assert Solution().convertDateToBinary("2000-01-01") == "11111010000-1-1"

# Bài 2
assert Solution().convertDateToBinary("1999-12-25") == "11111001111-1100-11001"
assert int("11111001111", 2) == 1999 and int("11001", 2) == 25


# Bài 3
def convert_todo_ans(date):
    return "-".join(f"{int(p):b}" for p in date.split("-"))


for d, expected in examples:
    assert convert_todo_ans(d) == expected


# Bài 4
def to_bin_rec(n):
    if n < 2:
        return str(n)
    return to_bin_rec(n // 2) + str(n % 2)


def convert_recursive(date):
    return "-".join(to_bin_rec(int(p)) for p in date.split("-"))


for d, expected in examples:
    assert convert_recursive(d) == expected
assert convert_recursive("2100-12-31") == Solution().convertDateToBinary("2100-12-31")


# Bài 5(a)
def binary_to_date_ans(binary):
    y, m, d = (int(p, 2) for p in binary.split("-"))
    return f"{y:04d}-{m:02d}-{d:02d}"


assert binary_to_date_ans("100000100000-10-11101") == "2080-02-29"
assert binary_to_date_ans("11101101100-1-1") == "1900-01-01"


# Bài 5(b)
def is_palindrome_binary_date(date):
    year_bin = f"{int(date[:4]):b}"
    return year_bin == year_bin[::-1]


assert is_palindrome_binary_date("1900-01-01") is False        # 11101101100 không đối xứng
assert is_palindrome_binary_date("2047-05-05") is True         # 2047 = 11111111111 (11 bit 1)
assert is_palindrome_binary_date("1911-03-03") is True          # 1911 = 11101110111 đối xứng
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tách → số → nhị phân → ghép)

```python
class Solution:
    def convertDateToBinary(self, date: str) -> str:
        return "-".join(f"{int(p):b}" for p in date.split("-"))
```

**Checklist**

- [ ] Đã `split("-")` thành ba phần năm/tháng/ngày?
- [ ] Đã `int(...)` **trước** khi đổi nhị phân (để bỏ số `0` đầu và tránh `TypeError`)?
- [ ] Không còn tiền tố `0b` (dùng `[2:]` hoặc `:b`)?
- [ ] Đổi nhị phân **cả số**, không phải từng chữ số?
- [ ] Ghép lại bằng `-` đúng thứ tự năm-tháng-ngày?
- [ ] Đã thử `"2080-02-29"` ra `"100000100000-10-11101"` và `"1900-01-01"` ra `"11101101100-1-1"`?
- [ ] Skeleton đúng `class Solution` / `def convertDateToBinary(self, date: str) -> str`?